# Phase 1: download, trim, extract (Kaggle)

Downloads one SO-101 dataset from Hugging Face, cuts every episode at the gripper-close moment, and stores per episode: DINOv2 patch features (for the baseline arm), raw 160x160 frames (for the pixel MAE), and joints at 10 Hz.

**Before running:** Settings (right panel) → Accelerator: GPU T4 x2 or P100; Internet: On.

**After running:** Save Version → the `/kaggle/working/features/<name>` folder becomes this notebook's output, which later notebooks attach as input data.

Run once per dataset by changing `REPO_ID` / `NAME` in the config cell. grasp_1 first (~3 GB, ~15 min), then grasp_2 (~11 GB, ~1 h).

In [ ]:
REPO_ID = "5hadytru/so101_grasp_1"   # then "5hadytru/so101_grasp_2"
NAME = "grasp_1"
SMOKE_TEST_EPISODES = None            # set to 3 for a quick check first

GIT_REPO = "https://github.com/yashica-patodia/so101-imitation-learning.git"

In [ ]:
# Optional: use a Hugging Face token stored in Kaggle Secrets (Add-ons > Secrets, label HF_TOKEN)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("no HF_TOKEN secret, downloading anonymously (slower):", type(e).__name__)

In [ ]:
import os; os.chdir("/kaggle/working"); os.makedirs("/kaggle/tmp", exist_ok=True)  # never delete the dir we stand in
!pip -q install av huggingface_hub pyarrow 2>&1 | tail -1
!rm -rf /kaggle/working/repo
!git clone {GIT_REPO} /kaggle/working/repo
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/tmp /kaggle/working | tail -2
!ls /kaggle/working/repo/nano_vla/data

In [ ]:
import os, subprocess, sys
assert os.path.isdir("/kaggle/working/repo/nano_vla"), "repo not cloned: run the cell above first and read its output"
os.chdir("/kaggle/working/repo")
cmd = [sys.executable, "-m", "nano_vla.data.extract",
       "--repo-id", REPO_ID,
       "--work", f"/kaggle/tmp/{NAME}",          # scratch: raw download lives here and is discarded
       "--out", f"/kaggle/working/features/{NAME}",  # kept as notebook output
       "--delete-videos"]
if SMOKE_TEST_EPISODES:
    cmd += ["--episodes", str(SMOKE_TEST_EPISODES)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Sanity check: one episode's frames at start / 1 s before close / at close / end, both cameras, plus sizes
import json, glob, numpy as np, matplotlib.pyplot as plt
out = f"/kaggle/working/features/{NAME}"
meta = json.load(open(f"{out}/meta.json"))
files = sorted(glob.glob(f"{out}/episode_*.npz"))
print(len(files), "episodes;", sum(os.path.getsize(f) for f in files)/1e9, "GB")
z = np.load(files[len(files)//2], allow_pickle=True)
tc, it = float(z["t_close"]), z["img_t"]; i = int(np.argmin(np.abs(it - tc)))
cams = meta["cam_short"]
fig, ax = plt.subplots(4, len(cams), figsize=(4*len(cams), 12))
for r, j in enumerate([0, max(0, i-5), i, len(it)-1]):
    for c, cam in enumerate(cams):
        a = ax[r][c] if len(cams) > 1 else ax[r]
        a.imshow(z["frm_"+cam][j]); a.set_title(f"{cam} t={it[j]:.1f}s"); a.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(8,3)); plt.plot(z["joint_t"], z["state"][:,5]); plt.axvline(tc, color="r"); plt.title("gripper joint, red = detected close"); plt.show()
L = [e["n_img"]/meta["img_hz"] for e in meta["episodes"] if "n_img" in e]
plt.hist(L, bins=30); plt.xlabel("segment length (s)"); plt.show()